# Week 2 — Weight matrix and first spiking simulation

**Step 1: full connection table.** Load the bulk Janelia export (segment-to-segment), keep only neuron-to-neuron connections for the 176,422 neurons in the Week 1 neuron table, and save as parquet. No synapse cutoff yet (that's tested in Week 3).

Memory budget: WSL is capped at 4 GB, so the filter runs in pyarrow without a pandas copy. Restart the kernel before running.

### Setup: paths and file check

In [1]:
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import pyarrow.compute as pc

DATA = Path.home() / 'arm_simulation/brain/data'   # on Colab, change only this line
RAW = DATA / 'raw'

F_WEIGHTS = RAW / 'connectome-weights-male-cns-v1.0-minconf-0.5.feather'
F_NT      = RAW / 'body-neurotransmitters-male-cns-v1.0.feather'
F_NEURONS = DATA / 'neurons_male-cns_v1.0.parquet'
F_CONN    = DATA / 'connections_male-cns_v1.0.parquet'   # output of Cell B

for p in [F_WEIGHTS, F_NT, F_NEURONS]:
    size = f"{p.stat().st_size / 1e6:,.1f} MB" if p.exists() else 'MISSING'
    print(f"{p.name:60s} {size}")

connectome-weights-male-cns-v1.0-minconf-0.5.feather         1,051.2 MB
body-neurotransmitters-male-cns-v1.0.feather                 43.3 MB
neurons_male-cns_v1.0.parquet                                2.7 MB


### Cell A: inspect the files (nothing big is loaded)

In [2]:
dset = ds.dataset(F_WEIGHTS, format='ipc')
print(dset.schema)
print(f"{dset.count_rows():,} segment-to-segment rows")

neurons = pd.read_parquet(F_NEURONS)
print(f"\nneuron table: {len(neurons):,} rows")
print(list(neurons.columns))

nt = ds.dataset(F_NT, format='ipc')
print(f"\nneurotransmitter table: {nt.count_rows():,} rows")
print(nt.schema)

body_pre: int64
body_post: int64
weight: int64
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 604
151,856,684 segment-to-segment rows

neuron table: 176,422 rows
['bodyId', 'type', 'instance', 'pre', 'post', 'cls', 'subcls', 'nt', 'side']

neurotransmitter table: 1,835,518 rows
body: int64
cell_type: string
total_nt_predictions: int32
predicted_nt_confidence: double
predicted_nt: string
ground_truth: string
celltype_total_nt_predictions: int32
celltype_predicted_nt: string
celltype_predicted_nt_confidence: double
consensus_nt: string
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1606


### Cell B: filter to neuron-to-neuron connections and save

If Cell A shows different column names for the weights file, edit `PRE`, `POST`, `W` below. The cell skips the filter if the output already exists.

In [3]:
PRE, POST, W = 'body_pre', 'body_post', 'weight'   # rename if Cell A's schema differs

missing = [c for c in (PRE, POST, W) if c not in dset.schema.names]
assert not missing, f"columns not found: {missing}; available: {dset.schema.names}"
assert 'bodyId' in neurons.columns, f"no bodyId column; available: {list(neurons.columns)}"

if F_CONN.exists():
    print(f"{F_CONN.name} already exists; loading it instead of re-filtering")
    tbl = pq.read_table(F_CONN)
else:
    ids = pa.array(neurons['bodyId'].astype('int64').values)
    tbl = dset.to_table(columns=[PRE, POST, W],
                        filter=ds.field(PRE).isin(ids) & ds.field(POST).isin(ids))
    pq.write_table(tbl, F_CONN, compression='zstd')
    print(f"saved {F_CONN.name} ({F_CONN.stat().st_size / 1e6:,.0f} MB)")

count = lambda mask: pc.sum(pc.cast(mask, 'int64')).as_py()
print(f"neuron-to-neuron connections: {tbl.num_rows:,}")
print(f"total synapses: {pc.sum(tbl[W]).as_py():,}")
print(f">=5 synapses: {count(pc.greater_equal(tbl[W], 5)):,}")
print(f"self-connections: {count(pc.equal(tbl[PRE], tbl[POST])):,}")
print("\nWeek 1 reference: 25.9M connections, 6.3M with >=5 synapses")

connections_male-cns_v1.0.parquet already exists; loading it instead of re-filtering
neuron-to-neuron connections: 25,862,574
total synapses: 125,024,863
>=5 synapses: 6,287,789
self-connections: 122

Week 1 reference: 25.9M connections, 6.3M with >=5 synapses


---
## Step 2: signed weight matrix

Weight of a connection = synapse count × sign(presynaptic transmitter) × `w_syn`.

This step saves **signed synapse counts** (count × sign) without `w_syn`. The simulation multiplies by `w_syn` at run time, so the Step 5 sweep never has to rebuild the matrix.

Sign rules:
| transmitter | baseline sign | modulator-excitatory variant |
|---|---|---|
| acetylcholine | +1 | +1 |
| GABA, glutamate | −1 | −1 |
| histamine (photoreceptors) | −1 | −1 |
| dopamine, serotonin, octopamine | 0 | +1 |
| unclear / missing | majority of same cell type, else +1 (flagged as guessed); histamine fill only in optic lobe | same |

Output folder `weights_male-cns_v1.0/`: sign table, indexed edge list, sparse matrix, and a metadata file. Every later week loads from here.

### Cell C: transmitter labels in the neuron table

In [4]:
import numpy as np, json

N = len(neurons)
assert neurons['bodyId'].is_unique, "duplicate bodyIds in neuron table"

nt_raw = neurons['nt'].astype('string').str.strip().str.lower()
print(nt_raw.value_counts(dropna=False).to_string())

FAST = {'acetylcholine': +1, 'gaba': -1, 'glutamate': -1, 'histamine': -1}
MOD  = {'dopamine', 'serotonin', 'octopamine', 'tyramine'}
UNKNOWN = {'unclear', 'none', 'nan', ''}

unexpected = set(nt_raw.dropna().unique()) - set(FAST) - MOD - UNKNOWN
print(f"\nunexpected labels: {unexpected or 'none'}")

nt
acetylcholine    95409
glutamate        28199
unclear          22902
gaba             20357
dopamine          4447
<NA>              2257
histamine         2242
serotonin          483
octopamine         126

unexpected labels: none


### Cell D: build the sign table

If Cell C printed unexpected labels, tell me before running this. They would be treated as unknown and guessed.

In [5]:
known = nt_raw.isin(set(FAST) | MOD)

# majority transmitter per cell type, using only neurons with a known label
typed = neurons['type'].notna() & known
majority = (pd.DataFrame({'type': neurons['type'][typed], 'nt': nt_raw[typed]})
              .groupby('type')['nt'].agg(lambda s: s.value_counts().idxmax()))
from_type = neurons['type'].map(majority)

# histamine is a photoreceptor transmitter: only allow it as a cell-type fill in the optic lobe.
# Elsewhere (e.g. leg motor neurons whose types got a histamine majority) fall back to the default.
HIST_OK = {'ol_sensory', 'ol_intrinsic'}
bad_hist = (from_type == 'histamine') & ~neurons['cls'].isin(HIST_OK)
print(f"histamine fills rejected outside the optic lobe: {(bad_hist & ~known).sum():,}")
from_type = from_type.mask(bad_hist)

nt_used = nt_raw.where(known, from_type).fillna('acetylcholine')
nt_source = np.select([known, from_type.notna()], ['predicted', 'celltype_majority'], 'default_ach')

signs = pd.DataFrame({
    'idx':        np.arange(N, dtype=np.int32),      # Brian2 neuron index, fixed from here on
    'bodyId':     neurons['bodyId'].astype('int64').values,
    'type':       neurons['type'].values,
    'cls':        neurons['cls'].values,
    'side':       neurons['side'].values,
    'nt_raw':     nt_raw.values,
    'nt_used':    nt_used.values,
    'nt_source':  nt_source,
    'guessed':    ~known.values,
})
signs['sign']        = signs['nt_used'].map(FAST).fillna(0).astype(np.int8)           # modulators -> 0
signs['sign_modexc'] = np.where(signs['nt_used'].isin(MOD), 1, signs['sign']).astype(np.int8)

print(signs['nt_source'].value_counts().to_string(), '\n')
print(pd.crosstab(signs['nt_used'], signs['sign']).to_string(), '\n')
print(f"guessed neurons: {signs['guessed'].sum():,}")
print("guessed, by what they were filled with:")
print(signs[signs.guessed]['nt_used'].value_counts().to_string())

histamine fills rejected outside the optic lobe: 65
nt_source
predicted            151263
celltype_majority     14366
default_ach           10793 

sign              -1     0       1
nt_used                           
acetylcholine      0     0  111437
dopamine           0  4459       0
gaba           21952     0       0
glutamate      29862     0       0
histamine       8031     0       0
octopamine         0   142       0
serotonin          0   539       0 

guessed neurons: 25,159
guessed, by what they were filled with:
nt_used
acetylcholine    16028
histamine         5789
glutamate         1663
gaba              1595
serotonin           56
octopamine          16
dopamine            12


### Cell E: edge list in Brian2 indices

In [6]:
idx_of = pd.Index(signs['bodyId'])
pre_idx  = idx_of.get_indexer(tbl[PRE].to_numpy()).astype(np.int32)
post_idx = idx_of.get_indexer(tbl[POST].to_numpy()).astype(np.int32)
assert (pre_idx >= 0).all() and (post_idx >= 0).all(), "connection to a neuron missing from the table"
count = tbl[W].to_numpy().astype(np.int32)

sign_pre = signs['sign'].to_numpy()[pre_idx]
syn_by_sign = pd.Series(count).groupby(sign_pre).sum()
for s, label in [(1, 'excitatory (+)'), (-1, 'inhibitory (-)'), (0, 'modulator (0)')]:
    n = int(syn_by_sign.get(s, 0))
    print(f"{label:16s} {n:>13,} synapses  ({n / count.sum():.1%})")

guess_pre = signs['guessed'].to_numpy()[pre_idx]
print(f"\nsynapses from guessed neurons: {count[guess_pre].sum() / count.sum():.1%}")

excitatory (+)      72,792,873 synapses  (58.2%)
inhibitory (-)      47,985,431 synapses  (38.4%)
modulator (0)        4,246,559 synapses  (3.4%)

synapses from guessed neurons: 2.4%


### Cell F: save the weight set

In [7]:
import scipy.sparse as sp

OUT = DATA / 'weights_male-cns_v1.0'
OUT.mkdir(exist_ok=True)

signs.to_parquet(OUT / 'neurons_signs.parquet')

edges = pa.table({'pre_idx': pre_idx, 'post_idx': post_idx, 'syn_count': count})
pq.write_table(edges, OUT / 'edges.parquet', compression='zstd')

# rows = postsynaptic, cols = presynaptic, so input to all neurons = W @ presynaptic_spikes
W_signed = sp.csr_matrix((count * sign_pre, (post_idx, pre_idx)), shape=(N, N), dtype=np.float32)
W_signed.eliminate_zeros()     # drops modulator edges (sign 0) from this baseline matrix
sp.save_npz(OUT / 'W_signed_counts.npz', W_signed)

meta = {
    'dataset': 'male-cns:v1.0',
    'source_file': F_WEIGHTS.name,
    'neurons': int(N),
    'edges_all': int(len(count)),
    'edges_in_baseline_matrix': int(W_signed.nnz),
    'total_synapses': int(count.sum()),
    'synapse_cutoff': None,
    'self_connections_kept': True,
    'matrix_layout': 'CSR, rows=post, cols=pre, values = syn_count * sign (no w_syn)',
    'sign_rules': {**FAST, **{m: 0 for m in MOD}},
    'sign_modexc': 'modulators = +1, otherwise as baseline',
    'unknown_fill': 'majority nt of same cell type (known neurons only), else acetylcholine',
    'histamine_fill_only_in': sorted(HIST_OK),
    'guessed_neurons': int(signs['guessed'].sum()),
}
(OUT / 'meta.json').write_text(json.dumps(meta, indent=2))

for p in sorted(OUT.iterdir()):
    print(f"{p.name:28s} {p.stat().st_size / 1e6:8.1f} MB")
print(f"\nmatrix: {W_signed.shape}, {W_signed.nnz:,} nonzero")

W_signed_counts.npz              79.9 MB
edges.parquet                    92.4 MB
meta.json                         0.0 MB
neurons_signs.parquet             2.9 MB

matrix: (176422, 176422), 24,287,791 nonzero


### Cell G: reload check

Reloads everything from disk and checks it against the in-memory version, plus one spot check on a JO-C/E neuron. Later weeks load exactly these files.

In [8]:
s2 = pd.read_parquet(OUT / 'neurons_signs.parquet')
e2 = pq.read_table(OUT / 'edges.parquet')
W2 = sp.load_npz(OUT / 'W_signed_counts.npz')

assert len(s2) == N and (s2['idx'].values == np.arange(N)).all()
assert e2.num_rows == len(count)
assert W2.nnz == W_signed.nnz and abs(W2).sum() == abs(W_signed).sum()

# spot check: outgoing connections of one JO-C/E neuron
jo = s2[s2['type'].astype(str).str.match(r'^JO-(C|E)')].iloc[0]
i = int(jo['idx'])
from_table = int(count[pre_idx == i].sum()) * int(jo['sign'])
from_matrix = int(W2[:, i].sum())
print(f"{jo['type']} (bodyId {jo['bodyId']}, {jo['nt_used']}): "
      f"table {from_table:,} vs matrix {from_matrix:,} signed synapses")
assert from_table == from_matrix
print("\nall checks passed")

JO-EV2 (bodyId 327202, acetylcholine): table 109 vs matrix 109 signed synapses

all checks passed


---
## Step 3: LIF model in Brian2

*(cells added after Step 2 output is checked)*